In [2]:
 
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# res='1km'
# Np_str='1e6'

# dx = 1 km; Np = 1M; Nt = 1 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_1min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_1min.nc') #***
res='1km'
Np_str='1e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [ ]:
#INITIALIZE DATA FUNCTION
###############################################################
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
def initiate_array(out_file,vars,t_chunk_size,p_chunk_size,t_size=None,p_size=None):
    # Define array dimensions (adjust based on your data)

    if t_size==None:
        t_size = len(data['time'])  # Number of timesteps
    if p_size==None:
        p_size = len(parcel['xh'])    # Number of vertical levels
    
    with h5py.File(out_file, 'w') as f: 
        # Check if the dataset 'theta_e' already exists
        for var_name in vars:
            if var_name not in f:
                # Create a dataset with the full size for all time steps (initially empty)
                f.create_dataset(var_name, 
                                 (t_size, p_size),  # Full size for all timesteps
                                 chunks=(t_chunk_size, p_chunk_size))  # Chunks for time axis to allow resizing

In [3]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [ ]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=180 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(data['time']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

In [ ]:
#Indexing Array with JobArray
data=data.isel(time=slice(start_job,end_job))
parcel=parcel.isel(time=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [2]:
###########################################################################################################################################################################

In [ ]:
#LOADING VARIABLES
###############################################################

In [13]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# open_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
open_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min.h5'
with h5py.File(open_file, 'r') as f:
    Z = f['Z'][start_job:end_job]
    Y = f['Y'][start_job:end_job]
    X = f['X'][start_job:end_job]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'FuncAnimation': '0.0 MB', 'PillowWriter': '0.0 MB', 'times': '0.0 MB'}

3.192 GB in use overall


In [14]:
def call_variable(varname):
    if varname=='th_e':
        with h5py.File(dir + 'Variable_Calculation/' + 'theta_e'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
            var_data = f['theta_e'][start_job:end_job]
    else:
        var_data=data[varname].data
    return var_data

In [15]:
def make_lagrangian_array(varname):
    var_data=call_variable(varname)
    VAR=np.zeros_like(Z,dtype='float32')
    
    Nt=len(data['time'])
    Np=len(parcel['xh'])
    for p in np.arange(Np):
        if np.mod(p,2e5)==0: print(f"{p}/{len(parcel['xh'])}")
    
        #Get Indicies
        zs=Z[:,p]
        ys=Y[:,p]
        xs=X[:,p]
        ts = np.arange(Nt)  
    
        #Get Values
        vars = var_data[ts, zs, ys, xs]

        #Adding to Lagrangian Array
        VAR[:,p]=vars

        del vars
    del var_data
    return VAR

In [ ]:
#W BUDGET VARIABLES
#########################################
WB_HADV=make_lagrangian_array('wb_hadv');check_memory()
WB_VADV=make_lagrangian_array('wb_vadv');check_memory()
WB_HIDIFF=make_lagrangian_array('wb_hidiff');check_memory()
WB_VIDIFF=make_lagrangian_array('wb_vidiff');check_memory()
WB_HTURB=make_lagrangian_array('wb_hturb');check_memory()
WB_VTURB=make_lagrangian_array('wb_vturb');check_memory()
WB_PGRAD=make_lagrangian_array('wb_pgrad');check_memory()
WB_BUOY=make_lagrangian_array('wb_buoy');check_memory()

In [ ]:
# Saving Data
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}.h5', 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('WB_HADV', data=WB_HADV)
    f.create_dataset('WB_VADV', data=WB_VADV)
    f.create_dataset('WB_HIDIFF', data=WB_HIDIFF)
    f.create_dataset('WB_VIDIFF', data=WB_VIDIFF)
    f.create_dataset('WB_HTURB', data=WB_HTURB)
    f.create_dataset('WB_VTURB', data=WB_VTURB)
    f.create_dataset('WB_PGRAD', data=WB_PGRAD)
    f.create_dataset('WB_BUOY', data=WB_BUOY)
del WB_HADV,WB_VADV,WB_HIDIFF,WB_VIDIFF,WB_HTURB,WB_VTURB,WB_PGRAD,WB_BUOY
check_memory()

In [ ]:
# Saving Data
##############
print('Saving Data\n')
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
# out_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_5min_{job_id}.h5'
out_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_{job_id}.h5'

vars=['W_HADV','W_VADV','WB_HIDIFF','WB_VIDIFF','WB_HTURB','WB_VTURB','WB_PGRAD','WB_BUOY']
initiate_array(out_file,vars,t_chunk_size=3,p_chunk_size=500)

with h5py.File(out_file, 'a') as f: 
    f['WB_HADV'][:]=WB_HADV
    f['WB_VADV'][:]=WB_VADV
    f['WB_HIDIFF'][:]=WB_HIDIFF
    f['WB_VIDIFF'][:]=WB_VIDIFF
    f['WB_HTURB'][:]=WB_HTURB
    f['WB_VTURB'][:]=WB_VTURB
    f['WB_PGRAD'][:]=WB_PGRAD
    f['WB_BUOY'][:]=WB_BUOY

In [ ]:
#QV BUDGET VARIABLES
#########################################
QVB_HADV=make_lagrangian_array('qvb_hadv');check_memory()
QVB_VADV=make_lagrangian_array('qvb_vadv');check_memory()
QVB_HIDIFF=make_lagrangian_array('qvb_hidiff');check_memory()
QVB_VIDIFF=make_lagrangian_array('qvb_vidiff');check_memory()
QVB_HTURB=make_lagrangian_array('qvb_hturb');check_memory()
QVB_VTURB=make_lagrangian_array('qvb_vturb');check_memory()
QVB_MP=make_lagrangian_array('qvb_mp');check_memory()

In [18]:
# Saving Data
##############
print('Saving Data\n')
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
# out_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_5min_{job_id}.h5'
out_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_{job_id}.h5'

vars=['QVB_HADV','QVB_VADV','QVB_HIDIFF','QVB_VIDIFF','QVB_HTURB','QVB_VTURB','QVB_MP']
initiate_array(out_file,vars,t_chunk_size=3,p_chunk_size=500)

with h5py.File(out_file, 'a') as f: 
    f['QVB_HADV'][:]=QVB_HADV
    f['QVB_VADV'][:]=QVB_VADV
    f['QVB_HIDIFF'][:]=QVB_HIDIFF
    f['QVB_VIDIFF'][:]=QVB_VIDIFF
    f['QVB_HTURB'][:]=QVB_HTURB
    f['QVB_VTURB'][:]=QVB_VTURB
    f['QVB_MP'][:]=QVB_MP

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'times': '0.0 MB', 'dir2': '0.0 MB', 'open': '0.0 MB'}

3.192 GB in use overall


In [ ]:
#TH BUDGET VARIABLES
#########################################
PTB_HADV=make_lagrangian_array('ptb_hadv');check_memory()
PTB_VADV=make_lagrangian_array('ptb_vadv');check_memory()
PTB_HIDIFF=make_lagrangian_array('ptb_hidiff');check_memory()
PTB_VIDIFF=make_lagrangian_array('ptb_vidiff');check_memory()
PTB_HTURB=make_lagrangian_array('ptb_hturb');check_memory()
PTB_VTURB=make_lagrangian_array('ptb_vturb');check_memory()
PTB_MP=make_lagrangian_array('ptb_mp');check_memory()
PTB_RAD=make_lagrangian_array('ptb_rad');check_memory()
PTB_DIV=make_lagrangian_array('ptb_div');check_memory()
PTB_DISS=make_lagrangian_array('ptb_diss');check_memory()

In [ ]:
# Saving Data
##############
print('Saving Data\n')
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
# out_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_5min_{job_id}.h5'
out_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_{job_id}.h5'

vars=['PTB_HADV','PTB_VADV','PTB_HIDIFF','PTB_VIDIFF','PTB_HTURB','PTB_VTURB','PTB_MP',
      'PTB_RAD','PTB_DIV','PTB_DISS']
initiate_array(out_file,vars,t_chunk_size=3,p_chunk_size=500)

with h5py.File(out_file, 'a') as f: 
    f['THB_HADV'][:]=THB_HADV
    f['THB_VADV'][:]=THB_VADV
    f['THB_HIDIFF'][:]=THB_HIDIFF
    f['THB_VIDIFF'][:]=THB_VIDIFF
    f['THB_HTURB'][:]=THB_HTURB
    f['THB_VTURB'][:]=THB_VTURB
    f['THB_MP'][:]=THB_MP
    f['THB_RAD'][:]=THB_RAD
    f['THB_DIV'][:]=THB_DIV
    f['THB_DISS'][:]=THB_DISS

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOB ARRAY IS RUNNING
# recombine=True

In [ ]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
    dir3=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
    # out_file=dir3+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_5min.h5'
    out_file=dir3+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_1min.h5'
    
    vars=['W_HADV','W_VADV','WB_HIDIFF','WB_VIDIFF','WB_HTURB','WB_VTURB','WB_PGRAD','WB_BUOY']
    initiate_array(out_file,vars,t_chunk_size=50,p_chunk_size=1000)
    
    with h5py.File(out_file, 'r+') as f_out:
        num_jobs=60
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,5)==0: print(f"job_id = {job_id}")
            [a,b] = get_job_range(job_id,num_jobs)
            
            # in_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_5min_{job_id}.h5'
            in_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_{job_id}.h5'
            with h5py.File(in_file, 'r') as f_in: 
                for var in vars:
                    f_out[var][a:b]=f_in[var][:]

In [ ]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
    dir3=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
    # out_file=dir3+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_5min.h5'
    out_file=dir3+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_1min.h5'
    
    vars=['QVB_HADV','QVB_VADV','QVB_HIDIFF','QVB_VIDIFF','QVB_HTURB','QVB_VTURB','QVB_MP']
    initiate_array(out_file,vars,t_chunk_size=50,p_chunk_size=1000)
    
    with h5py.File(out_file, 'r+') as f_out:
        num_jobs=60
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,5)==0: print(f"job_id = {job_id}")
            [a,b] = get_job_range(job_id,num_jobs)
            
            # in_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_5min_{job_id}.h5'
            in_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_{job_id}.h5'
            with h5py.File(in_file, 'r') as f_in: 
                for var in vars:
                    f_out[var][a:b]=f_in[var][:]

In [ ]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
    dir3=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
    # out_file=dir3+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_5min.h5'
    out_file=dir3+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_1min.h5'
    
    vars=['PTB_HADV','PTB_VADV','PTB_HIDIFF','PTB_VIDIFF','PTB_HTURB','PTB_VTURB','PTB_MP',
          'PTB_RAD','PTB_DIV','PTB_DISS']
    initiate_array(out_file,vars,t_chunk_size=50,p_chunk_size=1000)
    
    with h5py.File(out_file, 'r+') as f_out:
        num_jobs=60
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,5)==0: print(f"job_id = {job_id}")
            [a,b] = get_job_range(job_id,num_jobs)
            
            # in_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_5min_{job_id}.h5'
            in_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_{job_id}.h5'
            with h5py.File(in_file, 'r') as f_in: 
                for var in vars:
                    f_out[var][a:b]=f_in[var][:]

In [ ]:
#########################################

In [4]:
# import h5py
# # File path to the saved data
# dir2 = dir + 'Project_Algorithms/Lagrangian_Binary_Array/'
# filename = f'W_BUDGET_VARS_binary_array_{res}_{Np_str}.h5'

# # Open the HDF5 file and read the datasets
# with h5py.File(dir2 + filename, 'r') as f:
#     WB_HADV = f['WB_HADV'][:]
#     WB_VADV = f['WB_VADV'][:]
#     WB_HIDIFF = f['WB_HIDIFF'][:]
#     WB_VIDIFF = f['WB_VIDIFF'][:]
#     WB_HTURB = f['WB_HTURB'][:]
#     WB_VTURB = f['WB_VTURB'][:]
#     WB_PGRAD = f['WB_PGRAD'][:]
#     WB_BUOY = f['WB_BUOY'][:]

# # Check memory after loading
# check_memory()


Top 10 objects with highest memory usage
{'WB_HADV': '532.0 MB', 'WB_VADV': '532.0 MB', 'WB_HIDIFF': '532.0 MB', 'WB_VIDIFF': '532.0 MB', 'WB_HTURB': '532.0 MB', 'WB_VTURB': '532.0 MB', 'WB_PGRAD': '532.0 MB', 'WB_BUOY': '532.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB'}

4.256 GB in use overall


In [ ]:
# import h5py
# # File path to the saved data
# dir2 = dir + 'Project_Algorithms/Lagrangian_Binary_Array/'
# filename = f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}.h5'

# # Open the HDF5 file and read the datasets
# with h5py.File(dir2 + filename, 'r') as f:
#     QVB_HADV = f['QVB_HADV'][:]
#     QVB_VADV = f['QVB_VADV'][:]
#     QVB_HIDIFF = f['QVB_HIDIFF'][:]
#     QVB_VIDIFF = f['QVB_VIDIFF'][:]
#     QVB_HTURB = f['QVB_HTURB'][:]
#     QVB_VTURB = f['QVB_VTURB'][:]
#     QVB_MP = f['QVB_MP'][:]

# # Check memory after loading
# check_memory()


In [21]:
# import h5py
# # File path to the saved data
# dir2 = dir + 'Project_Algorithms/Lagrangian_Binary_Array/'
# filename = f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}.h5'

# # Open the HDF5 file and read the datasets
# with h5py.File(dir2 + filename, 'r') as f:
#     PTB_HADV = f['PTB_HADV'][:]
#     PTB_VADV = f['PTB_VADV'][:]
#     PTB_HIDIFF = f['PTB_HIDIFF'][:]
#     PTB_VIDIFF = f['PTB_VIDIFF'][:]
#     PTB_HTURB = f['PTB_HTURB'][:]
#     PTB_VTURB = f['PTB_VTURB'][:]
#     PTB_MP = f['PTB_MP'][:]
#     PTB_RAD = f['PTB_RAD'][:]
#     PTB_DIV = f['PTB_DIV'][:]
#     PTB_DISS = f['PTB_DISS'][:]

# # Check memory after loading
# check_memory()